In [ ]:
%pip install -q dotenv llama_stack_client==0.4.2

In [ ]:
import os
import requests
from io import BytesIO
from dotenv import load_dotenv

from llama_stack_client import LlamaStackClient

In [ ]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")

client = LlamaStackClient(base_url=base_url)

models = client.models.list()
[m for m in models if m.custom_metadata.get("model_type") == "embedding"]

In [ ]:
embedding_model = os.getenv("VDB_EMBEDDING", "sentence-transformers/ibm-granite/granite-embedding-125m-english")
embedding_dimension = int(os.getenv("VDB_EMBEDDING_DIMENSION", 768))

vs = client.vector_stores.create(
    name="hr-benefits-hybrid",
    extra_body={
        "embedding_model": embedding_model,
        "embedding_dimension": embedding_dimension,
        "search_mode": "hybrid",
        "bm25_weight": 0.5,
        "semantic_weight": 0.5,
    }
)

In [5]:
url = "https://raw.githubusercontent.com/burrsutter/fantaco-redhat-one-2026/refs/heads/main/rag-llama-stack/source_docs/FantaCoFabulousHRBenefits_clean.txt"

response = requests.get(url, timeout=30)
text_content = response.text

In [6]:
text_buffer = BytesIO(text_content.encode('utf-8'))
text_buffer.name = "hr-benefits-clean.txt"

uploaded_file = client.files.create(
    file=text_buffer,
    purpose="assistants"
)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1/files "HTTP/1.1 200 OK"


In [7]:
client.vector_stores.files.create(
    vector_store_id=vs.id,
    file_id=uploaded_file.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 100,
            "chunk_overlap_tokens": 10
        }
    }
)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1/vector_stores/vs_41a872f7-11de-4586-a91e-09900afd48ea/files "HTTP/1.1 200 OK"


VectorStoreFile(id='file-53d0ae56aa3c4d3a9c9e129befd3ca74', attributes={}, chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(chunk_overlap_tokens=10, max_chunk_size_tokens=100), type='static'), created_at=1771397926, object='vector_store.file', status='completed', usage_bytes=0, vector_store_id='vs_41a872f7-11de-4586-a91e-09900afd48ea', last_error=None)

In [ ]:
MODEL = "vllm/qwen3-8b"
INSTRUCTIONS = "You MUST use the knowledge_search tool to answer ALL questions by searching the provided documents."
VECTOR_STORE_ID = vs.id

TOOLS = [
    {
        "type": "function",
        "name": "knowledge_search",
        "description": "Search the HR benefits knowledge base for relevant information",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query to find relevant documents"}
            },
            "required": ["query"],
        },
    }
]

In [ ]:
def execute_rag_tool_call(response):
    """Execute knowledge_search function calls via vector_stores.search."""
    function_calls = []
    for item in response.output:
        if getattr(item, "type", None) == "function_call":
            function_calls.append(item)

    if not function_calls:
        return None

    import json
    tool_inputs = []
    for fc in function_calls:
        args = json.loads(fc.arguments) if isinstance(fc.arguments, str) else fc.arguments
        query = args.get("query", "")
        print(f"\n🔍 Searching: {query}")

        search_response = client.vector_stores.search(
            vector_store_id=VECTOR_STORE_ID,
            query=query,
            max_num_results=5,
        )
        results_text = "\n".join(
            f"[{i+1}] {r.content[0].text}" for i, r in enumerate(search_response.data) if r.content
        )
        print(f"📋 Found {len(search_response.data)} results")

        tool_inputs.append({
            "type": "function_call_output",
            "call_id": fc.call_id,
            "output": results_text if results_text else "No results found.",
        })

    return client.responses.create(
        model=MODEL,
        input=tool_inputs,
        instructions=INSTRUCTIONS,
        tools=TOOLS,
        stream=True,
        previous_response_id=response.id,
    )


def stream_rag(stream):
    """Stream responses with RAG tool call handling."""
    final_response = None

    for event in stream:
        event_type = getattr(event, "type", None)

        if event_type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
        elif event_type == "response.refusal.delta":
            print(event.delta, end="", flush=True)
        elif event_type == "response.completed":
            final_response = event.response

    print()

    if final_response:
        next_stream = execute_rag_tool_call(final_response)
        if next_stream:
            stream_rag(next_stream)


query = "What do I receive when I retire?"

stream = client.responses.create(
    model=MODEL,
    input=query,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
)

stream_rag(stream)